In [ ]:
pip install pyodbc

In [ ]:
import pyodbc
import csv
import os

def exportar_historico_por_dia():
    diretorio_atual = os.getcwd()

    connection_string = (
        r'DRIVER={ODBC Driver 17 for SQL Server};'
        r'SERVER=MeuServidor;'
        r'DATABASE=MeuDatabase;'
        r'Trusted_Connection=yes;'
    )

    query_datas = """
        SELECT DISTINCT CONVERT(DATE, InsertTime) AS Dia
        FROM dbo.HistoricoTamanhoTabela
        WHERE InsertTime >= '20250101'
        ORDER BY Dia
    """

    query_dados = """
        SELECT 
            RowId,
            l1,
            InsertTime,
            'DB_'  + LEFT(CONVERT(VARCHAR(64),
                        HASHBYTES('SHA2_256', ISNULL([Database], '')),
                        2), 8)              AS [Database],
            [Schema],
            'TBL_' + LEFT(CONVERT(VARCHAR(64),
                        HASHBYTES('SHA2_256', ISNULL([Table], '')),
                        2), 8)              AS [Table],
            row_count,
            CONCAT_WS('.',
                'DB_'  + LEFT(CONVERT(VARCHAR(64), HASHBYTES('SHA2_256', ISNULL([Database], '')), 2), 8),
                [Schema],
                'TBL_' + LEFT(CONVERT(VARCHAR(64), HASHBYTES('SHA2_256', ISNULL([Table],    '')), 2), 8)
            ) AS Objeto
        FROM dbo.HistoricoTamanhoTabela
        WHERE CAST(InsertTime AS DATE) = ?
    """

    print("Conectando ao banco de dados MeuServidor...")

    try:
        with pyodbc.connect(connection_string) as conexao:

            # 1. Busca a lista de datas disponíveis
            with conexao.cursor() as cursor_datas:
                cursor_datas.execute(query_datas)
                datas = [row.Dia for row in cursor_datas.fetchall()]

            print(f"{len(datas)} dia(s) encontrado(s). Iniciando exportação...\n")

            # 2. Para cada data, exporta um CSV
            with conexao.cursor() as cursor_dados:
                for dia in datas:
                    dia_str = dia.strftime('%Y-%m-%d')
                    nome_arquivo = f'HistoricoTamanhoTabela_{dia_str}.csv'
                    caminho_completo = os.path.join(diretorio_atual, nome_arquivo)

                    cursor_dados.execute(query_dados, dia)
                    colunas = [col[0] for col in cursor_dados.description]
                    linhas = cursor_dados.fetchall()

                    if linhas:
                        with open(caminho_completo, mode='w', newline='', encoding='utf-8') as arquivo_csv:
                            writer = csv.writer(arquivo_csv, delimiter=';')
                            writer.writerow(colunas)
                            writer.writerows(linhas)
                        print(f"  {dia_str} → {len(linhas):>6} linhas  →  {nome_arquivo}")
                    else:
                        print(f"  {dia_str} → sem dados, arquivo não gerado")

        print("\nExportação concluída.")

    except pyodbc.Error as erro_bd:
        print(f"Erro de Banco de Dados: {erro_bd}")
    except Exception as erro_geral:
        print(f"Erro inesperado: {erro_geral}")


if __name__ == "__main__":
    exportar_historico_por_dia()